# Phần 4: Khai phá insight và kể chuyện bằng dữ liệu
## Nghiên cứu và trực quan hóa các đặc trưng âm thanh và dự đoán xu hướng bài hát thịnh hành trên Spotify


### Mục tiêu và ý đồ kể chuyện
Âm nhạc không tồn tại trong chân không; nó là tấm gương phản chiếu chân thực bối cảnh kinh tế, sự biến động tâm lý xã hội và các làn sóng văn hóa toàn cầu. Cuốn notebook này được thiết kế để trả lời **5 câu hỏi nghiên cứu cốt lõi**:
1. **Nghịch lý kinh tế và cảm xúc:** Liệu các quốc gia giàu có với GDP cao có xu hướng nghe nhạc vui vẻ, lạc quan hơn các quốc gia đang phát triển?
2. **Biến động 3 pha đại dịch COVID-19:** Đại dịch thế kỷ đã định hình và chuyển dịch trạng thái tâm lý âm nhạc của nhân loại ra sao qua các năm từ 2020 đến 2022?
3. **Bản đồ tâm lý Russell 4 góc phần tư:** Người nghe toàn cầu phân bổ vào đâu trên ma trận cảm xúc và năng lượng, và một ca khúc cần yếu tố gì để trở thành hiện tượng?
4. **Sự sụp đổ của rào cản ngôn ngữ:** Làn sóng âm nhạc Latin và K-Pop đã vươn lên thách thức sự thống trị của tiếng Anh như thế nào?
5. **Giải phẫu công thức vàng của một bản hit:** Các ca khúc lọt top 50 có những đặc tính âm học nội tại nào khác biệt, và hiện tượng TikTokization rút ngắn thời lượng bài hát ra sao?

In [1]:
# 1. Khởi tạo môi trường phân tích và cấu hình đồ họa tương tác chuẩn mực
import os
import sys
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Cấu hình chuẩn cho biểu đồ tương tác Plotly
pio.templates.default = "plotly_white"
pio.renderers.default = "notebook_connected"

# Tránh lỗi font và hỗ trợ tiếng Việt trên console Windows
if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8")
    except Exception:
        pass

warnings.filterwarnings("ignore")

# Thiết lập đường dẫn thư mục dự án
cwd = os.getcwd()
PROJECT_ROOT = os.path.dirname(cwd) if os.path.basename(cwd) == "notebooks" else cwd
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data_loader import AUDIO_FEATURES, load_master_data, load_ml_data

print(f"Thư mục dự án: {PROJECT_ROOT}")
print(f"Phiên bản thư viện: Pandas {pd.__version__} | NumPy {np.__version__} | Plotly {plotly.__version__}")
print("Môi trường đã sẵn sàng cho hành trình phân tích và kể chuyện dữ liệu tương tác!")


Thư mục dự án: d:\LAPTRINH\TUONGTACDATA\PROJECT
Phiên bản thư viện: Pandas 2.3.3 | NumPy 2.4.0 | Plotly 6.9.0
Môi trường đã sẵn sàng cho hành trình phân tích và kể chuyện dữ liệu tương tác!


## 0. Nạp tập dữ liệu hợp nhất (1,078,125 dòng × 42 cột)

Sử dụng hàm chuẩn hóa `load_master_data()` từ module `src/data_loader.py` để nạp toàn bộ dữ liệu 74 quốc gia và bảng xếp hạng toàn cầu Global giai đoạn 2020 – 2022 tích hợp các chỉ số kinh tế vĩ mô từ World Bank.

In [2]:
# Nạp dữ liệu master hợp nhất từ data/processed/
df = load_master_data(PROJECT_ROOT)

# Đảm bảo các cột alias và bổ trợ sẵn sàng cho phân tích
if "artist_individual" in df.columns and "artists" not in df.columns:
    df["artists"] = df["artist_individual"]
elif "artists" in df.columns and "artist_individual" not in df.columns:
    df["artist_individual"] = df["artists"]

print(f"Quy mô tập dữ liệu: {df.shape[0]:,} dòng × {df.shape[1]} cột")
print(f"Số quốc gia ghi nhận: {df['country'].nunique()} (bao gồm các quốc gia và bảng Global)")
print(f"Khung thời gian: từ tuần {df['week'].min().strftime('%Y-%m-%d')} đến tuần {df['week'].max().strftime('%Y-%m-%d')}")
print(f"Tổng lượt nghe trực tuyến tích lũy: {df['streams'].sum():,} lượt")


Đã nạp master: 1,078,125 dòng, 42 cột
Quy mô tập dữ liệu: 1,078,125 dòng × 42 cột
Số quốc gia ghi nhận: 74 (bao gồm 74 quốc gia và bảng Global)
Khung thời gian: từ tuần 2020-01-02 đến tuần 2022-07-14
Tổng lượt nghe trực tuyến tích lũy: 504,712,062,748 lượt


## Nghịch lý kinh tế và cảm xúc

### Biểu đồ 1: Tương quan của cả 9 đặc trưng âm học với Mức độ Hạnh phúc

#### Câu hỏi được trả lời
*Trong 9 đặc trưng âm học của Spotify, đặc trưng nào liên hệ mạnh nhất với mức độ hạnh phúc quốc gia?


In [ ]:
# ==============================================================================
# Biểu đồ 1: Tương quan của cả 9 đặc trưng âm học với Điểm Hạnh phúc (WHR)
# Loại biểu đồ: Diverging horizontal bar chart (Plotly tương tác)
# ==============================================================================
import os
import sys
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import plotly.graph_objects as go
from IPython.display import display

# Nạp dữ liệu mart_country_year từ data/processed/
mart_path = os.path.join(PROJECT_ROOT, "data", "processed", "mart_country_year.parquet")
df_mart = pd.read_parquet(mart_path)

# Danh sách 9 đặc trưng âm học chuẩn của Spotify
features = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]

feature_display = {
    'danceability': 'Danceability (Độ khiêu vũ)',
    'energy': 'Energy (Cường độ năng lượng)',
    'loudness': 'Loudness (Âm lượng)',
    'speechiness': 'Speechiness (Độ dày lời ca)',
    'acousticness': 'Acousticness (Độ mộc mạc)',
    'instrumentalness': 'Instrumentalness (Độ khí nhạc)',
    'liveness': 'Liveness (Độ sống động sân khấu)',
    'valence': 'Valence (Cảm xúc tích cực)',
    'tempo': 'Tempo (Nhịp độ BPM)'
}

# Hàm tính toán hệ số Spearman và phân loại màu nhấn theo ngưỡng ý nghĩa thống kê p < 0.05
def compute_diverging_data(year):
    sub = df_mart[df_mart['year'] == year]
    records = []
    for feat in features:
        col = f"{feat}_weighted"
        r, p = spearmanr(sub['happiness_score'], sub[col])
        sig = (p < 0.05)
        
        # Màu nhấn: Nổi bật rực rỡ khi p < 0.05, màu xám trung tính khi p >= 0.05
        if sig:
            color = "#1DB954" if r > 0 else "#E74C3C" # Spotify Green (Thuận) / Coral Red (Nghịch)
            border_color = "#15843C" if r > 0 else "#C0392B"
            status_text = "Có ý nghĩa thống kê (p < 0.05) ⭐"
        else:
            color = "#BDC3C7" # Muted Gray
            border_color = "#95A5A6"
            status_text = "Không có ý nghĩa thống kê (p ≥ 0.05)"
            
        direction = "Tương quan thuận (+)" if r > 0 else "Tương quan nghịch (-)"
        
        records.append({
            'feature': feat,
            'name': feature_display[feat],
            'r': r,
            'p': p,
            'sig': sig,
            'color': color,
            'border_color': border_color,
            'status': status_text,
            'direction': direction,
            'label': f" {r:+.3f} (p={p:.3f}){' *' if sig else ''} "
        })
    return pd.DataFrame(records)

# Sắp xếp theo r năm 2022 tăng dần để giá trị dương cao nhất nằm ở trên cùng
df_corr_2022 = compute_diverging_data(2022).sort_values('r', ascending=True).reset_index(drop=True)
fixed_order = df_corr_2022['name'].tolist()

# Năm 2021 theo cùng thứ tự để không bị xáo trộn vị trí các trục khi chuyển đổi
df_corr_2021 = compute_diverging_data(2021).set_index('name').loc[fixed_order].reset_index()

fig_diverging = go.Figure()

# Trace 0: Năm 2022 (Mặc định hiển thị)
fig_diverging.add_trace(go.Bar(
    y=df_corr_2022['name'],
    x=df_corr_2022['r'],
    orientation='h',
    name="Năm 2022",
    marker=dict(
        color=df_corr_2022['color'],
        line=dict(color=df_corr_2022['border_color'], width=1.5)
    ),
    text=df_corr_2022['label'],
    textposition="outside",
    textfont=dict(size=11, family="Arial", color="#2C3E50"),
    customdata=np.stack((
        df_corr_2022['p'],
        df_corr_2022['status'],
        df_corr_2022['direction']
    ), axis=-1),
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Năm khảo sát: 2022 (73 quốc gia)<br>"
        "Hệ số tương quan Spearman (r): <b>%{x:.3f}</b><br>"
        "Giá trị p-value: <b>%{customdata[0]:.4f}</b><br>"
        "Chiều hướng: %{customdata[2]}<br>"
        "Ý nghĩa thống kê: <b>%{customdata[1]}</b>"
        "<extra></extra>"
    ),
    visible=True
))

# Trace 1: Năm 2021
fig_diverging.add_trace(go.Bar(
    y=df_corr_2021['name'],
    x=df_corr_2021['r'],
    orientation='h',
    name="Năm 2021",
    marker=dict(
        color=df_corr_2021['color'],
        line=dict(color=df_corr_2021['border_color'], width=1.5)
    ),
    text=df_corr_2021['label'],
    textposition="outside",
    textfont=dict(size=11, family="Arial", color="#2C3E50"),
    customdata=np.stack((
        df_corr_2021['p'],
        df_corr_2021['status'],
        df_corr_2021['direction']
    ), axis=-1),
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Năm khảo sát: 2021 (68 quốc gia)<br>"
        "Hệ số Spearman (r): <b>%{x:.3f}</b><br>"
        "Giá trị p-value: <b>%{customdata[0]:.4f}</b><br>"
        "Chiều hướng: %{customdata[2]}<br>"
        "Ý nghĩa thống kê: <b>%{customdata[1]}</b>"
        "<extra></extra>"
    ),
    visible=False
))

# Cấu hình Layout chuẩn Diverging Bar Chart từ -1 đến +1
fig_diverging.update_layout(
    title=dict(
        text="<b>Biểu đồ 1: Tương quan Spearman giữa Mức độ Hạnh phúc (WHR) và 9 Đặc trưng Âm học Spotify</b><br>"
             "<sup>Năm 2022: Liveness (+0.366) & Tempo (+0.347) liên hệ thuận mạnh nhất; Acousticness (-0.281) nghịch mạnh nhất (p < 0.05). Valence (r = +0.064) không có tương quan.</sup>",
        x=0.02,
        y=0.96,
        font=dict(size=14, family="Arial")
    ),
    xaxis=dict(
        title="Hệ số tương quan hạng Spearman (r)",
        range=[-1.0, 1.0],
        dtick=0.2,
        zeroline=True,
        zerolinewidth=2.5,
        zerolinecolor="#2C3E50",
        gridcolor="#ECF0F1",
        tickformat="+.1f",
        tickfont=dict(size=11)
    ),
    yaxis=dict(
        title="",
        gridcolor="#ECF0F1",
        tickfont=dict(size=12)
    ),
    annotations=[
        dict(
            x=-0.5, y=-0.15, xref="x", yref="paper",
            text="◀ <b>TƯƠNG QUAN NGHỊCH</b> (Càng hạnh phúc, chỉ số càng thấp)",
            showarrow=False, font=dict(size=11, color="#C0392B")
        ),
        dict(
            x=0.5, y=-0.15, xref="x", yref="paper",
            text="<b>TƯƠNG QUAN THUẬN</b> (Càng hạnh phúc, chỉ số càng cao) ▶",
            showarrow=False, font=dict(size=11, color="#1DB954")
        )
    ],
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            active=0,
            x=0.02,
            y=1.12,
            xanchor="left",
            yanchor="top",
            bgcolor="#ffffff",
            bordercolor="#CBD5E0",
            borderwidth=1,
            buttons=[
                dict(
                    label="📅 Năm 2022 (73 quốc gia)",
                    method="update",
                    args=[
                        {"visible": [True, False]},
                        {
                            "title.text": "<b>Biểu đồ 1: Tương quan Spearman giữa Mức độ Hạnh phúc (WHR) và 9 Đặc trưng Âm học (Năm 2022)</b><br>"
                                          "<sup>Năm 2022: Liveness (+0.366, p=0.001) & Tempo (+0.347, p=0.003) tương quan thuận; Acousticness (-0.281, p=0.016) tương quan nghịch có ý nghĩa (p < 0.05).</sup>"
                        }
                    ]
                ),
                dict(
                    label="📅 Năm 2021 (68 quốc gia)",
                    method="update",
                    args=[
                        {"visible": [False, True]},
                        {
                            "title.text": "<b>Biểu đồ 1: Tương quan Spearman giữa Mức độ Hạnh phúc (WHR) và 9 Đặc trưng Âm học (Năm 2021)</b><br>"
                                          "<sup>Năm 2021: Chưa có đặc trưng nào vượt qua ngưỡng ý nghĩa p < 0.05; Danceability (r = -0.226, p = 0.064) và Liveness (r = +0.238, p = 0.051).</sup>"
                        }
                    ]
                )
            ]
        )
    ],
    showlegend=False,
    height=560,
    margin=dict(l=230, r=60, t=100, b=80)
)

fig_diverging.show()

# Bảng tổng hợp thống kê Spearman chi tiết của cả 9 đặc trưng năm 2022
summary_2022 = df_corr_2022[['name', 'r', 'p', 'sig', 'direction']].sort_values('r', ascending=False)
summary_2022.columns = ['Đặc trưng âm học', 'Spearman r', 'p-value', 'Có ý nghĩa (p < 0.05)', 'Chiều hướng']

print("Bảng tổng hợp tương quan Spearman 9 đặc trưng âm học với Điểm số Hạnh phúc (Năm 2022):")
display(summary_2022.round(4))